# Analisis Klaster Provinsi Indonesia (AHC & Median Imputation)

Notebook ini berisi alur kerja lengkap untuk melakukan analisis klaster pada data sosio-ekonomi 38 provinsi di Indonesia. Metodologi yang digunakan adalah **Agglomerative Hierarchical Clustering (AHC)** dan **Median Imputation** untuk menangani data yang hilang, sehingga seluruh 38 provinsi dapat dianalisis.

## 1. Impor Pustaka dan Pemuatan Data

Sel ini memuat semua pustaka yang diperlukan, membaca data dari `Data/Klaster Provinsi.csv`, dan melakukan imputasi median untuk mengisi nilai TPT yang hilang di 4 provinsi baru Papua.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Membaca data dari file CSV
try:
    df = pd.read_csv('Data/Klaster Provinsi.csv')
    print("Data berhasil dimuat dari 'Data/Klaster Provinsi.csv'.")
    
    # Menstandardisasi nama kolom untuk konsistensi
    df.rename(columns={
        'Tingkat NEET (%) (2024)': 'Tingkat NEET (%)',
        'TPT (%) (Feb 2023)': 'TPT (%)',
        'RLS (Tahun) (2024)': 'RLS (Tahun)',
        'PDRB per Kapita (Juta Rp) (2023)': 'PDRB per Kapita (Juta Rp)'
    }, inplace=True)
    
    # Konversi 'N/A' atau string kosong ke NaN
    features = ['Tingkat NEET (%)', 'TPT (%)', 'RLS (Tahun)', 'PDRB per Kapita (Juta Rp)']
    for col in features:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Imputasi missing value dengan median
    for col in features:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
    
    df_model = df.copy()

    # Memilih fitur dan melakukan normalisasi
    X = df_model[features]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"\nData 38 provinsi siap diproses dan dinormalisasi untuk pemodelan.")
    display(df_model.tail()) # Menampilkan data terakhir untuk cek imputasi

except FileNotFoundError:
    print("Error: File 'Data/Klaster Provinsi.csv' tidak ditemukan.")

## 2. Menentukan Jumlah Klaster Optimal

Kita akan menggunakan **Silhouette Score** untuk mengevaluasi seberapa baik pemisahan antar klaster untuk berbagai jumlah klaster. Skor yang lebih tinggi menunjukkan klaster yang lebih padat dan terpisah dengan baik.

In [ ]:
# Tuning jumlah klaster dengan Silhouette Score
range_n_clusters = range(2, 11)
silhouette_scores = []
for n_clusters in range_n_clusters:
    ahc = AgglomerativeClustering(n_clusters=n_clusters)
    cluster_labels = ahc.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(score)

# Menampilkan hasil tuning
plt.figure(figsize=(8,5))
plt.plot(range_n_clusters, silhouette_scores, marker='o')
plt.title('Silhouette Score untuk Berbagai Jumlah Klaster (AHC)')
plt.xlabel('Jumlah Klaster')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

## 3. Menjalankan AHC dan Menganalisis Hasil

Berdasarkan analisis dan tujuan *storytelling*, kita akan menggunakan **k=4** sebagai jumlah klaster final. Model AHC akan dijalankan dan hasilnya akan dianalisis untuk mendefinisikan setiap arketipe.

In [ ]:
# Menjalankan AHC dengan jumlah klaster = 4
optimal_k = 4
ahc = AgglomerativeClustering(n_clusters=optimal_k)
df_model['Klaster'] = ahc.fit_predict(X_scaled)

# Menghitung karakteristik rata-rata klaster
cluster_analysis = df_model.groupby('Klaster')[features].mean().round(2)

print("Karakteristik Rata-rata per Klaster (AHC):")
display(cluster_analysis)

print("\nDaftar Provinsi per Klaster (AHC):")
for i in sorted(df_model['Klaster'].unique()):
    print(f"\n--- Klaster {i} ---")
    provinces_in_cluster = df_model[df_model['Klaster'] == i]['Provinsi'].tolist()
    print(", ".join(provinces_in_cluster))

## 4. Visualisasi Peta Sebaran Klaster

Visualisasi ini akan memetakan setiap provinsi ke dalam klasternya masing-masing, memberikan gambaran geografis dari arketipe masalah NEET di Indonesia.

In [ ]:
# Menambahkan hasil klaster ke DataFrame asli untuk pemetaan
df_map = df.merge(df_model[['Provinsi', 'Klaster']], on='Provinsi', how='left')
df_map['Klaster'] = df_map['Klaster'].apply(lambda x: f'Klaster {int(x)}')

fig_map = px.choropleth(
    df_map,
    geojson="https://raw.githubusercontent.com/superpikar/indonesia-geojson/master/indonesia.geojson",
    locations='Provinsi',
    featureidkey='properties.state',
    color='Klaster',
    color_discrete_sequence=px.colors.qualitative.Plotly, # Menggunakan palet warna default
    scope='asia',
    title='<b>Peta Sebaran Klaster Provinsi Indonesia (Hasil AHC)</b>'
)
fig_map.update_geos(fitbounds="locations", visible=False)
fig_map.show()